# 06. ML: Churn Prediction
Предсказываем отмену подписки. Logistic Regression → Random Forest → XGBoost → SHAP.

In [ ]:
!pip install xgboost shap -q

In [ ]:
import pandas as pd
import os
import kagglehub
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, confusion_matrix
from xgboost import XGBClassifier

dataset_path = kagglehub.dataset_download('radistaleks/synthetic-bank-transactions')
categories    = pd.read_csv(os.path.join(dataset_path, 'categories.csv'))
clients       = pd.read_csv(os.path.join(dataset_path, 'clients.csv'))
subscriptions = pd.read_csv(os.path.join(dataset_path, 'subscriptions.csv'))
transactions  = pd.read_csv(os.path.join(dataset_path, 'transactions.csv'))

In [ ]:
clients['registration_date'] = pd.to_datetime(clients['registration_date'])
clients['birthdate']         = pd.to_datetime(clients['birthdate'])
subscriptions['date_start']  = pd.to_datetime(subscriptions['date_start'])
subscriptions['date_end']    = pd.to_datetime(subscriptions['date_end'])
transactions['date']         = pd.to_datetime(transactions['date'], format='%Y-%m-%d %H:%M:%S')

clients = clients.fillna(0)
subscriptions['product_company'] = subscriptions['product_company'].fillna('Неизвестно')
transactions['product_company']  = transactions['product_company'].fillna('Неизвестно')

SNAPSHOT = pd.Timestamp('2020-12-31')

## 1. Целевая переменная и выборка
`is_churned = 1` если подписка отменена в 2020 году.

In [ ]:
# берём подписки, начатые до 2020, с известным исходом
target = subscriptions[
    (subscriptions['date_start'] < pd.Timestamp('2020-01-01')) &
    (subscriptions['date_end'].isna() | (subscriptions['date_end'].dt.year == 2020))
].copy()

target['is_churned']    = target['date_end'].notna().astype(int)
target['duration_days'] = (target['date_end'].fillna(SNAPSHOT) - target['date_start']).dt.days

print('Подписок в выборке:', len(target))
print('Churn rate:        ', target['is_churned'].mean().round(3))

## 2. Feature Engineering

In [ ]:
# клиентские признаки
cf = clients[['id', 'gender', 'birthdate', 'registration_date', 'income', 'expenses', 'credit', 'deposit']].copy()
cf['age']           = (SNAPSHOT - cf['birthdate']).dt.days // 365
cf['months_reg']    = (SNAPSHOT - cf['registration_date']).dt.days // 30
cf['gender_F']      = (cf['gender'] == 'F').astype(int)
cf = cf.drop(columns=['gender', 'birthdate', 'registration_date'])
cf.head()

In [ ]:
# транзакционные признаки
tf = transactions.groupby('client_id').agg(
    total_spend   = ('amount', 'sum'),
    txn_count     = ('amount', 'count'),
    avg_check     = ('amount', 'mean'),
    n_categories  = ('product_category', 'nunique'),
    recency_days  = ('date', lambda x: (SNAPSHOT - x.max()).days)
).reset_index()

# доля выходных и ночных транзакций
weekend = transactions.assign(is_wknd=(transactions['date'].dt.dayofweek >= 5).astype(int))
tf['weekend_ratio'] = weekend.groupby('client_id')['is_wknd'].mean().values

night = transactions.assign(is_night=(transactions['date'].dt.hour < 6).astype(int))
tf['night_ratio'] = night.groupby('client_id')['is_night'].mean().values

tf.head()

In [ ]:
# признаки подписки
target['is_music']   = (target['product_category'] == 4).astype(int)
target['sub_amount'] = target['amount']

company_dummies = pd.get_dummies(target['product_company'], prefix='co', drop_first=True)
sf = pd.concat([target[['client_id', 'is_churned', 'is_music', 'sub_amount', 'duration_days']], company_dummies], axis=1)
sf.head()

In [ ]:
# собираем всё вместе
df = (
    sf
    .merge(cf.rename(columns={'id': 'client_id'}), on='client_id', how='left')
    .merge(tf, on='client_id', how='left')
    .dropna()
)
print('Shape:', df.shape)
print('Churn rate:', df['is_churned'].mean().round(3))

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['is_churned', 'client_id'])
y = df['is_churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Train churn:', y_train.mean().round(3), '| Test churn:', y_test.mean().round(3))

## 4. Logistic Regression

In [ ]:
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr_sc, y_train)

y_prob_lr = lr.predict_proba(X_te_sc)[:, 1]
print('ROC-AUC:', roc_auc_score(y_test, y_prob_lr).round(4))
print(classification_report(y_test, lr.predict(X_te_sc)))

## 5. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_prob_rf = rf.predict_proba(X_test)[:, 1]
print('ROC-AUC:', roc_auc_score(y_test, y_prob_rf).round(4))
print(classification_report(y_test, rf.predict(X_test)))

In [ ]:
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15).sort_values(
).plot(kind='barh', figsize=(10, 6), title='Random Forest: Feature Importance')

## 6. XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)

y_prob_xgb = xgb.predict_proba(X_test)[:, 1]
print('ROC-AUC:', roc_auc_score(y_test, y_prob_xgb).round(4))
print(classification_report(y_test, xgb.predict(X_test)))

## 7. Сравнение моделей

In [ ]:
pd.DataFrame({
    'Model':   ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'ROC-AUC': [roc_auc_score(y_test, y_prob_lr).round(4),
                roc_auc_score(y_test, y_prob_rf).round(4),
                roc_auc_score(y_test, y_prob_xgb).round(4)]
}).sort_values('ROC-AUC', ascending=False).set_index('Model')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name, prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf), ('XGBoost', y_prob_xgb)]:
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
ax.set_title('ROC Curves')

In [ ]:
cm = confusion_matrix(y_test, xgb.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Активен', 'Churned'], yticklabels=['Активен', 'Churned'])
plt.title('Confusion Matrix — XGBoost')

## 8. SHAP

In [ ]:
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15)

In [ ]:
shap.summary_plot(shap_values, X_test, max_display=15)

In [ ]:
# waterfall для самого рискованного клиента
idx = y_prob_xgb.argmax()
shap.waterfall_plot(shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx],
    feature_names=X_test.columns.tolist()
))

## 9. Клиенты под риском оттока

In [ ]:
# применяем модель ко всем активным подпискам
active = subscriptions[subscriptions['date_end'].isna()].copy()
active['is_music']    = (active['product_category'] == 4).astype(int)
active['sub_amount']  = active['amount']
active['duration_days'] = (SNAPSHOT - active['date_start']).dt.days

co_dummies = pd.get_dummies(active['product_company'], prefix='co')
af = pd.concat([active[['client_id', 'is_music', 'sub_amount', 'duration_days']], co_dummies], axis=1)

af = af.merge(cf.rename(columns={'id': 'client_id'}), on='client_id', how='left')
af = af.merge(tf, on='client_id', how='left').dropna()

# выравниваем колонки
for col in X.columns:
    if col not in af.columns:
        af[col] = 0
af = af[['client_id'] + list(X.columns)]

af['churn_proba'] = xgb.predict_proba(af[X.columns])[:, 1]
at_risk = af[['client_id', 'churn_proba']].sort_values('churn_proba', ascending=False).head(20)
at_risk.round(3)

In [ ]:
af['churn_proba'].hist(bins=20, figsize=(10, 4), edgecolor='white')
plt.title('Вероятность оттока для активных подписок')
plt.xlabel('Churn probability')